# MolDeTr — quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smidooo/MolDeTr/blob/main/notebooks/MolDeTr_quickstart.ipynb)

Detect the multiplets in a 1-D ¹H NMR window and read off chemical shift (δ), largest coupling (max J), and proton count in one pass. Runs on the free CPU runtime.

> **Research code accompanying the paper.** MolDeTr extracts δ, proton count, and couplings from real ¹H NMR spectra — including the congested, strongly-coupled cases it was built for — across 80–600 MHz. Predictions can deviate for inputs outside its trained regime: unusual distortions, non-standard pulse sequences or processing, mixtures/impurities, or regions wider than the 1200 Hz window. `max J` is the dominant coupling per multiplet (the full set is in `structured_output/`). Sanity-check against your chemistry. See [`docs/SCOPE.md`](https://github.com/smidooo/MolDeTr/blob/main/docs/SCOPE.md).

## 1. Install

In [ ]:
!pip -q install "moldetr[app] @ git+https://github.com/smidooo/MolDeTr.git" huggingface_hub

## 2. Get the checkpoint and an example spectrum

The trained weights (~974 MB) live on Hugging Face Hub / Zenodo, not in git. Adjust `REPO`/`FILE` if you host them elsewhere.

In [ ]:
from huggingface_hub import hf_hub_download

REPO, FILE = "smidooo/moldetr", "model_spin_system_ABCDEFG_exp2.pth"
ckpt = hf_hub_download(REPO, FILE)   # or download from Zenodo: 10.5281/zenodo.21217102

# The bundled vanillin example window (300 MHz):
!wget -q https://raw.githubusercontent.com/smidooo/MolDeTr/main/examples/roi_S8_example.npz
print("checkpoint:", ckpt)

## 3. Predict

In [ ]:
import os
import numpy as np
import pandas as pd
import moldetr
from moldetr.inference import build_model, load_checkpoint, run
from moldetr.postprocess import decode_predictions, load_extrema
from moldetr.visualization import plot_spectrum
from moldetr.validation import validate_spectrum

d = np.load("roi_S8_example.npz", allow_pickle=True)
amp = validate_spectrum(d["spectrum_padded"], points_per_hz=5.12)
axis = np.asarray(d["ppm_axis_padded"], dtype=float)
extrema = load_extrema(os.path.join(os.path.dirname(moldetr.__file__), "assets", "extrema.txt"))

model = load_checkpoint(build_model(), ckpt)
preds = decode_predictions(run(model, amp), extrema, 5.12,
                           ppm_left=float(axis[0]), ppm_right=float(axis[-1]))
fig, rows = plot_spectrum(amp, preds, ppm_left=float(axis[0]), ppm_right=float(axis[-1]))
display(pd.DataFrame(rows))
fig

## Your own data

MolDeTr expects one window: **6144 points at 5.12 points/Hz** (a 1200 Hz window), real-valued. See [`docs/INPUT_FORMAT.md`](https://github.com/smidooo/MolDeTr/blob/main/docs/INPUT_FORMAT.md) to prepare a region, and [`docs/USAGE_NOTES.md`](https://github.com/smidooo/MolDeTr/blob/main/docs/USAGE_NOTES.md) for how to read the output and its failure modes.